# 01 · ItemKNN on MovieLens-1M and Yelp

Thesis §5.1 · Tables 4–5 · Figures 4–5

Item-based KNN on the binary train interaction matrix. The neighbourhood size `topN` is tuned on
validation NDCG@10 (HR@10 breaks ties), then HR@K / NDCG@K are computed on the test split for K ∈ {5, 10, 20}.

* **Part 1 – MovieLens:** adjusted-cosine similarity.
* **Part 2 – Yelp:** plain cosine similarity.

> **Known issue.** `predict_weighted_sum` returns `Σ s·r / Σ |s|` with `r = 1`. With non-negative
> (cosine) similarities that is `1.0` for every item that shares a user with the history, so many
> candidates tie. `eval_hr_ndcg_fast` counts only *strictly* higher scores, so every tie goes to the
> held-out item and HR/NDCG come out optimistic (mainly the Yelp part). The thesis formula
> `ŷ_ui = Σ_{j∈N_K(i)∩I_u} s(i,j)` with pessimistic tie-breaking is in `src/cfcompare/knn.py`
> (`python scripts/run_experiment.py --model itemknn --dataset yelp`).
>
> The final MovieLens run in the thesis selected `topN = 200`. The grid below stops at 100.

In [ ]:
# Folder containing ml-1m.ratings.dat and yelp.rating (see data/README.md).
# On Kaggle, point this at the attached dataset, e.g. "/kaggle/input/<dataset-name>".
DATA_DIR = "../data"

In [ ]:
#FORRR MovieLens

import pandas as pd
import numpy as np
import random
import math
from collections import defaultdict
from functools import lru_cache

# ---------------------------------------------------------
# 1. Load MovieLens and build train / val / test splits
# ---------------------------------------------------------
df = pd.read_csv(
    f"{DATA_DIR}/ml-1m.ratings.dat",
    sep="::",
    engine="python",
    names=["user_id", "item_id", "rating", "timestamp"]
)

print("Original data:")
print(df.head())

user_counts = df.groupby("user_id").size()
print("Minimum number of ratings per user:", user_counts.min())

# sort by (user, timestamp)
df = df.sort_values(["user_id", "timestamp"])

# Split into train/test: last interaction per user -> test
train_rows = []
test_rows  = []

for user_id, group in df.groupby("user_id"):
    g = group.values
    test_rows.append(g[-1])       # last as test
    for row in g[:-1]:
        train_rows.append(row)    # others as train

train_df = pd.DataFrame(train_rows, columns=df.columns)
test_df  = pd.DataFrame(test_rows, columns=df.columns)

print("Train size (before val split):", len(train_df))
print("Test size:", len(test_df))

# Validation: from train_df, take last per user as val
train_df = train_df.sort_values(["user_id", "timestamp"])
val_df   = train_df.groupby("user_id").tail(1).copy()
train_df_cv = train_df.drop(val_df.index).copy()

print("Train size (after val split):", len(train_df_cv))
print("Val size:", len(val_df))

# ---------------------------------------------------------
# 2. Build implicit interaction matrix (0/1) from train_df_cv
# ---------------------------------------------------------
train_df_cv = train_df_cv.copy()
train_df_cv["interaction"] = 1

interaction_matrix = train_df_cv.pivot_table(
    index="user_id",
    columns="item_id",
    values="interaction",
    fill_value=0
)

interaction_matrix = interaction_matrix.astype(np.float32)
interaction_array = interaction_matrix.values

num_users, num_items = interaction_matrix.shape

# ---- mappings & caches for evaluation ----
user_index_list = interaction_matrix.index.tolist()
item_index_list = interaction_matrix.columns.tolist()

user2pivot = {uid: idx for idx, uid in enumerate(user_index_list)}
item2pivot = {iid: idx for idx, iid in enumerate(item_index_list)}

n_users, n_items = interaction_array.shape
user_item_dict = {
    u: set(np.where(interaction_array[u] == 1)[0])
    for u in range(n_users)
}
all_items_set = set(range(n_items))

num_ones = interaction_array.sum()
sparsity = 1 - (num_ones / (num_users * num_items))

print("\nUser-Item Interaction Matrix (train only):")
print(interaction_matrix.head())
print("Number of ones (interactions):", int(num_ones))
print("Number of users:", num_users)
print("Number of items:", num_items)
print("Sparsity: {:.2f}%".format(100 * sparsity))

# ---------------------------------------------------------
# 3. Similarity: adjusted cosine on implicit matrix
# ---------------------------------------------------------
R = interaction_matrix  # shorthand

# user-wise mean interaction rate (used only as fallback / centering)
user_mean   = R.mean(axis=1)
global_mean = float(R.values.mean())

# Precompute L2 norms of item columns (not strictly needed for adjusted cosine,
# but kept here if you want to switch back to plain cosine later)
item_norms = np.sqrt((R ** 2).sum(axis=0))  # Series: index=item_id


@lru_cache(maxsize=None)
def adjusted_cosine_similarity(item_i, item_j):
    """
    Adjusted cosine similarity between items i and j using mean-centered
    implicit interactions (0/1) per user.
    """
    if item_i not in R.columns or item_j not in R.columns:
        return 0.0

    xi = R[item_i]
    xj = R[item_j]

    # With implicit data and no NaNs, mask is all True, but we keep it for generality
    mask = xi.notna() & xj.notna()
    if mask.sum() == 0:
        return 0.0

    di = (xi[mask] - user_mean[mask])
    dj = (xj[mask] - user_mean[mask])

    num = float((di * dj).sum())
    den = float(np.sqrt((di**2).sum()) * np.sqrt((dj**2).sum()))
    return num / den if den > 0 else 0.0


# ---------------------------------------------------------
# 4. ItemKNN prediction (weighted sum over neighbors)
# ---------------------------------------------------------
def predict_weighted_sum(u, i, topN=50):
    """
    Item-based CF with implicit feedback.
    u, i are original user_id and item_id (as in R's index/columns).
    Returns a score (used only for ranking).
    """
    # Cold-start user
    if u not in R.index:
        return global_mean

    # Cold-start item
    if i not in R.columns:
        return float(user_mean.get(u, global_mean))

    # Items this user interacted with (R[u, j] > 0)
    rated = R.loc[u]
    rated = rated[rated > 0]
    if rated.empty:
        return float(user_mean.get(u, global_mean))

    sims = []
    for j, r in rated.items():
        s = adjusted_cosine_similarity(i, j)
        if np.isfinite(s) and s != 0.0:
            # r is 1 in implicit setting, but keep it explicit
            sims.append((abs(s), s, r))

    if not sims:
        return float(user_mean.get(u, global_mean))

    # keep strongest neighbors
    if topN is not None and len(sims) > topN:
        sims.sort(reverse=True, key=lambda t: t[0])
        sims = sims[:topN]

    num = sum(s * r for _, s, r in sims)
    den = sum(abs(s) for _, s, _ in sims)

    if den <= 1e-12:
        return float(user_mean.get(u, global_mean))

    pred = num / den
    # optional: clip to [0, 1] to be "probability-like"
    return float(np.clip(pred, 0.0, 1.0))


# ---------------------------------------------------------
# 5. Model wrapper with .predict([users_idx, items_idx])
# ---------------------------------------------------------
class ItemKNNModel:
    def __init__(self, R, user_index_list, item_index_list, topN=10):
        self.R = R
        self.user_index_list = user_index_list  # list of user_ids in pivot order
        self.item_index_list = item_index_list  # list of item_ids in pivot order
        self.topN = topN

    def predict(self, inputs, verbose=0):
        users_idx, items_idx = inputs  # indices in pivot space (0..n_users-1, 0..n_items-1)
        out = np.empty(len(items_idx), dtype=np.float32)
        for k, (ui, mj) in enumerate(zip(users_idx, items_idx)):
            u = self.user_index_list[ui]   # original user_id
            i = self.item_index_list[mj]   # original item_id
            out[k] = predict_weighted_sum(u, i, topN=self.topN)
        return out


# ---------------------------------------------------------
# 6. Fast evaluation: HR@K and NDCG@K
# ---------------------------------------------------------
def eval_hr_ndcg_fast(
    eval_df,
    model,
    user2pivot,
    item2pivot,
    user_item_dict,
    all_items_set,
    number_of_rec_items,
    max_test_size=10_000,
    neg_sample_size=100,
    batch_size=4096,
):
    """
    Fast evaluation of HR@K and NDCG@K by negative sampling.

    - eval_df: DataFrame with columns ['user_id', 'item_id'] (held-out positives)
    - model:  .predict([user_idx_array, item_idx_array])
    - user2pivot, item2pivot: maps original ids -> pivot indices
    - user_item_dict: {pivot_user: set(pivot_item)} from TRAIN
    - all_items_set: set of all pivot item indices
    """

    users = eval_df["user_id"].values
    items = eval_df["item_id"].values

    batch_users = []
    batch_items = []
    cand_lengths = []

    num_eval = 0

    for df_user, df_item in zip(users, items):
        # skip unknown users/items
        if df_user not in user2pivot or df_item not in item2pivot:
            continue

        u_idx = user2pivot[df_user]
        i_idx = item2pivot[df_item]

        items_interacted = user_item_dict.get(u_idx, set())

        # candidate negatives = all_items \ (positives ∪ {gt})
        possible_neg = list(all_items_set - items_interacted - {i_idx})
        if not possible_neg:
            continue

        k_neg = min(neg_sample_size, len(possible_neg))
        neg_samples = random.sample(possible_neg, k_neg)

        # put ground-truth FIRST, then negatives
        candidates = [i_idx] + neg_samples

        batch_users.append(np.full(len(candidates), u_idx, dtype=np.int32))
        batch_items.append(np.array(candidates, dtype=np.int32))
        cand_lengths.append(len(candidates))

        num_eval += 1
        if num_eval >= max_test_size:
            break

    if num_eval == 0:
        return 0.0, 0.0

    flat_users = np.concatenate(batch_users)
    flat_items = np.concatenate(batch_items)

    preds = model.predict([flat_users, flat_items], batch_size=batch_size, verbose=0).flatten()

    cursor = 0
    num_hits = 0.0
    sum_ndcg = 0.0

    for L in cand_lengths:
        user_preds = preds[cursor: cursor + L]
        cursor += L

        # ground-truth is at index 0
        gt_score = user_preds[0]
        # rank = #items with strictly higher score than GT (0-based)
        rank = np.sum(user_preds > gt_score)

        if rank < number_of_rec_items:
            num_hits += 1.0
            sum_ndcg += 1.0 / math.log2(rank + 2)  # rank 0 -> log2(2)

    hr_k = num_hits / num_eval
    ndcg_k = sum_ndcg / num_eval
    return hr_k, ndcg_k


Original data:
   user_id  item_id  rating  timestamp
0        1     1193       5  978300760
1        1      661       3  978302109
2        1      914       3  978301968
3        1     3408       4  978300275
4        1     2355       5  978824291
Minimum number of ratings per user: 20
Train size (before val split): 994169
Test size: 6040
Train size (after val split): 988129
Val size: 6040

User-Item Interaction Matrix (train only):
item_id  1     2     3     4     5     6     7     8     9     10    ...  \
user_id                                                              ...   
1         1.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...   
2         0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...   
3         0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...   
4         0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0   0.0  ...   
5         0.0   0.0   0.0   0.0   0.0   1.0   0.0   0.0   0.0   0.0  ...   

item_id  3943  3944  3945  39

In [ ]:
user_index_list = interaction_matrix.index.tolist()
item_index_list = interaction_matrix.columns.tolist()

candidate_topN = [5, 10, 20, 50, 100]
K_metric = 10  # we tune using HR@10 / NDCG@10

best_cfg = {
    "topN": None,
    "hr": -1.0,
    "ndcg": -1.0
}

print("\n=== Hyperparameter search over topN (number of neighbors) ===")
for topN in candidate_topN:
    print(f"\n--- Evaluating topN = {topN} ---")
    model_val = ItemKNNModel(
        R=interaction_matrix,
        user_index_list=user_index_list,
        item_index_list=item_index_list,
        topN=topN
    )

    hr_val, ndcg_val = eval_hr_ndcg_fast(
        eval_df=val_df,
        model=model_val,
        user2pivot=user2pivot,
        item2pivot=item2pivot,
        user_item_dict=user_item_dict,
        all_items_set=all_items_set,
        number_of_rec_items=K_metric,
        max_test_size=5000,
        neg_sample_size=100
    )
    print(f"[VAL] topN={topN}, HR@{K_metric}={hr_val:.4f}, NDCG@{K_metric}={ndcg_val:.4f}")

    # select best by NDCG, tie-break by HR
    better_ndcg = ndcg_val > best_cfg["ndcg"]
    tie_ndcg_better_hr = (math.isclose(ndcg_val, best_cfg["ndcg"]) and hr_val > best_cfg["hr"])
    if better_ndcg or tie_ndcg_better_hr:
        best_cfg["topN"] = topN
        best_cfg["hr"]   = hr_val
        best_cfg["ndcg"] = ndcg_val
        print(f"--> New best topN = {topN}")

print("\nBest topN (neighbors) on validation:")
print(best_cfg)

# ---------------------------------------------------------
# 8. Final evaluation on TEST for K ∈ {5, 10, 20}
# ---------------------------------------------------------
if best_cfg["topN"] is None:
    print("Warning: no valid topN found; cannot evaluate on test.")
else:
    print("\n=== Final test evaluation with best topN ===")
    best_topN = best_cfg["topN"]

    best_model = ItemKNNModel(
        R=interaction_matrix,
        user_index_list=user_index_list,
        item_index_list=item_index_list,
        topN=best_topN
    )

    for K_eval in [5, 10, 20]:
        print(f"\n[TEST] Evaluating HR@{K_eval}, NDCG@{K_eval} with topN={best_topN}")
        hr_t, ndcg_t = eval_hr_ndcg_fast(
            eval_df=test_df,
            model=best_model,
            user2pivot=user2pivot,
            item2pivot=item2pivot,
            user_item_dict=user_item_dict,
            all_items_set=all_items_set,
            number_of_rec_items=K_eval,
            max_test_size=10_000,
            neg_sample_size=100
        )
        print(f"[TEST] HR@{K_eval} = {hr_t:.4f}, NDCG@{K_eval} = {ndcg_t:.4f}")




=== Hyperparameter search over topN (number of neighbors) ===

--- Evaluating topN = 5 ---


TypeError: eval_hr_ndcg() got an unexpected keyword argument 'eval_df'

For Yelp:

In [ ]:
import pandas as pd
import numpy as np
import random
import math
import os
import sys
from collections import defaultdict
from functools import lru_cache

def read_yelp_rating_file(file_path):
    # Check if the file exists
    if not os.path.exists(file_path):
        print(f"Error: File '{file_path}' does not exist.")
        sys.exit(1)

    try:
        # Read the file into a DataFrame (auto-detects whitespace or tab)
        df = pd.read_csv(
            file_path,
            sep=r'\s+',            # handles both tabs and spaces
            header=None,
            names=['user_id', 'item_id', 'rating', 'timestamp'],
            engine='python'        # allows regex separator
        )

        # Display basic information about the DataFrame
        print("✅ Data successfully loaded.\n")
        print("DataFrame Info:")
        print(df.info())
        print("\nFirst 5 rows of the DataFrame:")
        print(df.head())

        return df

    except Exception as e:
        print(f" Error reading file '{file_path}': {e}")
        sys.exit(1)

yelp_file_path = f"{DATA_DIR}/yelp.rating"
yelp_df = read_yelp_rating_file(yelp_file_path)
df = yelp_df

user_counts = df.groupby("user_id").size()
print("Minimum number of ratings per user:", user_counts.min())

# sort by (user, timestamp)
df = df.sort_values(["user_id", "timestamp"])

# Split into train/test: last interaction per user -> test
train_rows = []
test_rows  = []

for user_id, group in df.groupby("user_id"):
    g = group.values
    test_rows.append(g[-1])       # last as test
    for row in g[:-1]:
        train_rows.append(row)    # others as train

train_df = pd.DataFrame(train_rows, columns=df.columns)
test_df  = pd.DataFrame(test_rows, columns=df.columns)

print("Train size (before val split):", len(train_df))
print("Test size:", len(test_df))

# Validation: from train_df, take last per user as val
train_df = train_df.sort_values(["user_id", "timestamp"])
val_df   = train_df.groupby("user_id").tail(1).copy()
train_df_cv = train_df.drop(val_df.index).copy()

print("Train size (after val split):", len(train_df_cv))
print("Val size:", len(val_df))

# ---------------------------------------------------------
# 2. Build implicit interaction matrix (0/1) from train_df_cv
# ---------------------------------------------------------
train_df_cv = train_df_cv.copy()
train_df_cv["interaction"] = 1

interaction_matrix = train_df_cv.pivot_table(
    index="user_id",
    columns="item_id",
    values="interaction",
    fill_value=0
)

interaction_matrix = interaction_matrix.astype(np.float32)
interaction_array = interaction_matrix.values

num_users, num_items = interaction_matrix.shape

# ---- mappings & caches for evaluation ----
user_index_list = interaction_matrix.index.tolist()
item_index_list = interaction_matrix.columns.tolist()

user2pivot = {uid: idx for idx, uid in enumerate(user_index_list)}
item2pivot = {iid: idx for idx, iid in enumerate(item_index_list)}

n_users, n_items = interaction_array.shape
user_item_dict = {
    u: set(np.where(interaction_array[u] == 1)[0])
    for u in range(n_users)
}
all_items_set = set(range(n_items))

num_ones = interaction_array.sum()
sparsity = 1 - (num_ones / (num_users * num_items))

print("\nUser-Item Interaction Matrix (train only):")
print(interaction_matrix.head())
print("Number of ones (interactions):", int(num_ones))
print("Number of users:", num_users)
print("Number of items:", num_items)
print("Sparsity: {:.2f}%".format(100 * sparsity))

# ---------------------------------------------------------
# 3. Similarity: binary cosine on interaction matrix
# ---------------------------------------------------------
R = interaction_matrix  # shorthand

# user-wise mean interaction rate (used only as fallback, not in similarity itself)
user_mean   = R.mean(axis=1)
global_mean = float(R.values.mean())

# Precompute L2 norms of item columns for cosine
item_norms = np.sqrt((R ** 2).sum(axis=0))  # Series: index=item_id


@lru_cache(maxsize=None)
def cosine_similarity(item_i, item_j):
    """
    Cosine similarity between items i and j using 0/1 interaction matrix.
    """
    if item_i not in R.columns or item_j not in R.columns:
        return 0.0

    xi = R[item_i]
    xj = R[item_j]

    num = float((xi * xj).sum())  # common users
    den = float(item_norms[item_i] * item_norms[item_j])
    return num / den if den > 0 else 0.0


# ---------------------------------------------------------
# 4. ItemKNN prediction (weighted sum over neighbors)
# ---------------------------------------------------------
def predict_weighted_sum(u, i, topN=50):
    """
    Item-based CF with implicit feedback.
    u, i are original user_id and item_id (as in R's index/columns).
    Returns a score (used only for ranking).
    """
    # Cold-start user
    if u not in R.index:
        return global_mean

    # Cold-start item
    if i not in R.columns:
        return float(user_mean.get(u, global_mean))

    # Items this user interacted with (R[u, j] == 1)
    rated = R.loc[u]
    rated = rated[rated > 0]
    if rated.empty:
        return float(user_mean.get(u, global_mean))

    sims = []
    for j, r in rated.items():
        s = cosine_similarity(i, j)
        if np.isfinite(s) and s != 0.0:
            sims.append((abs(s), s, r))

    if not sims:
        return float(user_mean.get(u, global_mean))

    # keep strongest neighbors
    if topN is not None and len(sims) > topN:
        sims.sort(reverse=True, key=lambda t: t[0])
        sims = sims[:topN]

    num = sum(s * r for _, s, r in sims)
    den = sum(abs(s) for _, s, _ in sims)

    if den <= 1e-12:
        return float(user_mean.get(u, global_mean))

    pred = num / den
    return float(np.clip(pred, 0.0, 1.0))


# ---------------------------------------------------------
# 5. Model wrapper with .predict([users_idx, items_idx])
# ---------------------------------------------------------
class ItemKNNModel:
    def __init__(self, R, user_index_list, item_index_list, topN=10):
        self.R = R
        self.user_index_list = user_index_list  # list of user_ids in pivot order
        self.item_index_list = item_index_list  # list of item_ids in pivot order
        self.topN = topN

    def predict(self, inputs, verbose=0):
        users_idx, items_idx = inputs  # indices in pivot space
        out = np.empty(len(items_idx), dtype=np.float32)
        for k, (ui, mj) in enumerate(zip(users_idx, items_idx)):
            u = self.user_index_list[ui]   # original user_id
            i = self.item_index_list[mj]   # original item_id
            out[k] = predict_weighted_sum(u, i, topN=self.topN)
        return out


# ---------------------------------------------------------
# 6. Fast evaluation: HR@K and NDCG@K
# ---------------------------------------------------------
def eval_hr_ndcg_fast(
    eval_df,
    model,
    user2pivot,
    item2pivot,
    user_item_dict,
    all_items_set,
    number_of_rec_items,
    max_test_size=10_000,
    neg_sample_size=100,
    batch_size=4096,
):
    """
    Fast evaluation of HR@K and NDCG@K by negative sampling.
    (Same logic as in MovieLens block.)
    """
    users = eval_df["user_id"].values
    items = eval_df["item_id"].values

    batch_users = []
    batch_items = []
    cand_lengths = []

    num_eval = 0

    for df_user, df_item in zip(users, items):
        if df_user not in user2pivot or df_item not in item2pivot:
            continue

        u_idx = user2pivot[df_user]
        i_idx = item2pivot[df_item]

        items_interacted = user_item_dict.get(u_idx, set())
        possible_neg = list(all_items_set - items_interacted - {i_idx})
        if not possible_neg:
            continue

        k_neg = min(neg_sample_size, len(possible_neg))
        neg_samples = random.sample(possible_neg, k_neg)

        candidates = [i_idx] + neg_samples

        batch_users.append(np.full(len(candidates), u_idx, dtype=np.int32))
        batch_items.append(np.array(candidates, dtype=np.int32))
        cand_lengths.append(len(candidates))

        num_eval += 1
        if num_eval >= max_test_size:
            break

    if num_eval == 0:
        return 0.0, 0.0

    flat_users = np.concatenate(batch_users)
    flat_items = np.concatenate(batch_items)

    preds = model.predict([flat_users, flat_items], batch_size=batch_size, verbose=0).flatten()

    cursor = 0
    num_hits = 0.0
    sum_ndcg = 0.0

    for L in cand_lengths:
        user_preds = preds[cursor: cursor + L]
        cursor += L

        gt_score = user_preds[0]
        rank = np.sum(user_preds > gt_score)

        if rank < number_of_rec_items:
            num_hits += 1.0
            sum_ndcg += 1.0 / math.log2(rank + 2)

    hr_k = num_hits / num_eval
    ndcg_k = sum_ndcg / num_eval
    return hr_k, ndcg_k

In [ ]:
# ---------------------------------------------------------
# 7. Hyperparameter tuning: topN (neighbors) using validation
# ---------------------------------------------------------
user_index_list = interaction_matrix.index.tolist()
item_index_list = interaction_matrix.columns.tolist()

candidate_topN = [5, 10, 20, 50, 100]
K_metric = 10  # we tune using HR@10 / NDCG@10

best_cfg = {
    "topN": None,
    "hr": -1.0,
    "ndcg": -1.0
}

print("\n=== Hyperparameter search over topN (number of neighbors) ===")
for topN in candidate_topN:
    print(f"\n--- Evaluating topN = {topN} ---")
    model_val = ItemKNNModel(
        R=interaction_matrix,
        user_index_list=user_index_list,
        item_index_list=item_index_list,
        topN=topN
    )

    hr_val, ndcg_val = eval_hr_ndcg_fast(
        eval_df=val_df,
        model=model_val,
        user2pivot=user2pivot,
        item2pivot=item2pivot,
        user_item_dict=user_item_dict,
        all_items_set=all_items_set,
        number_of_rec_items=K_metric,
        max_test_size=5000,
        neg_sample_size=100
    )
    print(f"[VAL] topN={topN}, HR@{K_metric}={hr_val:.4f}, NDCG@{K_metric}={ndcg_val:.4f}")

    better_ndcg = ndcg_val > best_cfg["ndcg"]
    tie_ndcg_better_hr = (math.isclose(ndcg_val, best_cfg["ndcg"]) and hr_val > best_cfg["hr"])
    if better_ndcg or tie_ndcg_better_hr:
        best_cfg["topN"] = topN
        best_cfg["hr"]   = hr_val
        best_cfg["ndcg"] = ndcg_val
        print(f"--> New best topN = {topN}")

print("\nBest topN (neighbors) on validation:")
print(best_cfg)

# ---------------------------------------------------------
# 8. Final evaluation on TEST for K ∈ {5, 10, 20}
# ---------------------------------------------------------
if best_cfg["topN"] is None:
    print("Warning: no valid topN found; cannot evaluate on test.")
else:
    print("\n=== Final test evaluation with best topN ===")
    best_topN = best_cfg["topN"]

    best_model = ItemKNNModel(
        R=interaction_matrix,
        user_index_list=user_index_list,
        item_index_list=item_index_list,
        topN=best_topN
    )

    for K_eval in [5, 10, 20]:
        print(f"\n[TEST] Evaluating HR@{K_eval}, NDCG@{K_eval} with topN={best_topN}")
        hr_t, ndcg_t = eval_hr_ndcg_fast(
            eval_df=test_df,
            model=best_model,
            user2pivot=user2pivot,
            item2pivot=item2pivot,
            user_item_dict=user_item_dict,
            all_items_set=all_items_set,
            number_of_rec_items=K_eval,
            max_test_size=10_000,
            neg_sample_size=100
        )
        print(f"[TEST] HR@{K_eval} = {hr_t:.4f}, NDCG@{K_eval} = {ndcg_t:.4f}")